# Acoustic PDM — Phase 1: Notebook 01
## Data Acquisition, Verification & Dataset Inspection (MIMII Dataset)

**Project:** Acoustic Predictive Maintenance (Acoustic PDM)  
**Dataset:** [Hitachi MIMII Dataset](https://zenodo.org/record/3384388) — a publicly available benchmark for industrial machine sound anomaly detection.  
**Scope:** 4 industrial machine types — **Fan**, **Pump**, **Slider**, **Valve** — recorded at **6 dB Signal-to-Noise Ratio (SNR)** for **Machine ID 00**.

---

### What is this notebook about?
Before building any machine-learning model, we must first **acquire**, **verify**, and **understand** the raw data we will work with. This notebook performs the essential first step of every data-science pipeline: **data ingestion and quality assurance**.

### Objectives
1. **Environment & Path Discovery:** Automatically detect whether the notebook is running on **Kaggle** (cloud) or on a **local machine**, and set up all file paths accordingly.
2. **Audio File Integrity Verification:** Open a stratified sample of `.wav` files and check that they meet the expected physical parameters — **16,000 Hz** sample rate, **mono** channel, **16-bit PCM** encoding, and approximately **10-second** duration.
3. **Data Inventory:** Systematically count how many *Normal* (healthy) vs. *Anomalous* (faulty) audio clips exist for each of the 4 machine types.
4. **Summary Report Export:** Save two CSV reports — `indexed_dataset.csv` (full file catalog) and `data_summary.csv` (aggregated counts) — so that downstream notebooks can load the catalog directly instead of re-scanning the file system.

### Why does this matter?
If corrupted, missing, or mis-labelled audio files slip through, every subsequent step (feature extraction, model training, evaluation) will produce unreliable results. Running this notebook first guarantees a **clean, verified starting point**.

---
### Step 0: Environment Bootstrap — Path Discovery & Configuration Loading

**This cell must run first** before any other cell in the notebook.

#### What it does
1. **Writes a copy of `config.yaml`** into the Kaggle working directory (needed on Kaggle because the working filesystem starts empty; harmless when running locally).
2. **Detects the runtime environment** — Kaggle vs. local — by checking for the `/kaggle/input` directory.
3. **Resolves all project paths** (project root, raw-data directory, reports, processed data, models) from a single `PROJECT_ROOT` variable.
4. **Loads `configs/config.yaml`** into the Python dictionary `CFG` so that every subsequent cell can reference centralised hyperparameters instead of hardcoding values.
5. **Adds the `src/` directory to the Python path** so that custom utility modules can be imported anywhere in the notebook.

#### Why this design?
Centralising path logic in a single bootstrap cell means the rest of the notebook is **completely environment-agnostic** — the same code runs on Kaggle and on your laptop without any manual path editing.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Config & Environment Bootstrap
# ═══════════════════════════════════════════════════════════════
import os
os.makedirs('/kaggle/working/configs', exist_ok=True)

config_text = """\
# ============================================================
# Acoustic PDM — Centralized Pipeline Configuration
# ============================================================
audio:
  sample_rate: 16000
  channels: 1
  bit_depth: 16
  clip_duration_sec: 10

features:
  n_fft: 1024
  hop_length: 512
  n_mels: 128
  fmin: 0
  fmax: null
  power_to_db: true
  context_frames: 5

normalization:
  method: "zscore"
  epsilon: 1.0e-8

data:
  raw_dir: "data/raw"
  processed_dir: "data/processed"
  machine_types:
    - "fan"
    - "pump"
    - "slider"
    - "valve"
  machine_ids: ["id_00"]
  snr_levels: ["6_dB"]
  test_split: 0.1

kaggle:
  dataset_slug: "daisukelab/dc2020task2"

training:
  batch_size: 32
  learning_rate: 0.001
  weight_decay: 1.0e-5
  epochs: 50
  early_stopping_patience: 10
  random_seed: 42
  num_workers: 2

model_ae:
  latent_dim: 32
  encoder_channels: [1, 32, 64, 128]
  kernel_size: 3
  stride: 2
  padding: 1
  activation: "leaky_relu"

model_vae:
  latent_dim: 32
  beta: 1.0

evaluation:
  threshold_percentile: 95
  reports_dir: "reports"

paths:
  models_dir: "models"
  scripts_dir: "scripts"
  reports_dir: "reports"
"""

with open('/kaggle/working/configs/config.yaml', 'w') as f:
    f.write(config_text)
print("✓ config.yaml written")

import sys
import yaml
from pathlib import Path

ON_KAGGLE = os.path.exists("/kaggle/input")

if ON_KAGGLE:
    PROJECT_ROOT = Path("/kaggle/working")
    _kaggle_data = None
    for _root, _dirs, _ in os.walk('/kaggle/input'):
        if 'dc2020task2' in _dirs:
            _kaggle_data = os.path.join(_root, 'dc2020task2')
            break
    DATA_ROOT = Path(_kaggle_data) if _kaggle_data else Path('/kaggle/input/dc2020task2')
else:
    _cwd = Path(os.getcwd()).resolve()
    PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
    DATA_ROOT    = PROJECT_ROOT / "data" / "raw"

REPORTS_DIR   = PROJECT_ROOT / "reports"
CONFIGS_DIR   = PROJECT_ROOT / "configs"
MODELS_DIR    = PROJECT_ROOT / "models"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

config_path = CONFIGS_DIR / "config.yaml"
if config_path.exists():
    with open(config_path, "r") as f:
        CFG = yaml.safe_load(f)
    print(f"✓ Config loaded from: {config_path}")
else:
    CFG = {}
    print(f"⚠ Config not found at {config_path}, using defaults")

print(f"Environment:  {'Kaggle' if ON_KAGGLE else 'Local'}")
print(f"Project Root: {PROJECT_ROOT}")
print(f"Data Root:    {DATA_ROOT}")
print(f"Reports Dir:  {REPORTS_DIR}")
print(f"Data exists:  {DATA_ROOT.exists()}")

---
### Step 1: Import Required Libraries

This cell imports all Python packages used throughout the notebook:

| Library | Purpose |
|---|---|
| **os, glob, pathlib** | File-system navigation and recursive file search |
| **time** | Timing long operations |
| **soundfile (sf)** | Low-level WAV header inspection (sample rate, channels, bit depth, duration) |
| **librosa** | Audio loading and digital signal processing (DSP) |
| **numpy** | Numerical array operations |
| **pandas** | Tabular data manipulation and CSV export |
| **matplotlib** | Plotting charts and saving figures |
| **tqdm** | Progress bars for long loops |

It also sets `OUTPUT_DIR` to the resolved `REPORTS_DIR` from the bootstrap cell, so that all generated files are saved in a consistent location.

In [ ]:
import glob
import time
import soundfile as sf
import librosa
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Use the bootstrap-resolved reports directory
OUTPUT_DIR = str(REPORTS_DIR)
print(f"Reports output directory ready: {OUTPUT_DIR}")

---
### Step 2: Build the Active Pipeline Configuration Dictionary

Rather than scattering magic numbers throughout the notebook, we pull every key parameter from the centralised `config.yaml` file (loaded earlier as `CFG`) and store them in a flat Python dictionary called `CONFIG`.

| Parameter | Value | Meaning |
|---|---|---|
| `sample_rate` | 16,000 Hz | Number of audio samples captured per second; this is the native rate of the MIMII recordings |
| `channels` | 1 | Mono audio (single microphone channel) |
| `duration_sec` | 10.0 s | Expected length of each audio clip |
| `machines` | fan, pump, slider, valve | The 4 industrial machine types we study |
| `machine_id` | id_00 | We focus on a single machine unit per type for controlled comparison |
| `snr` | 6_dB | Signal-to-Noise Ratio level; 6 dB means the machine sound is ~4× louder than background factory noise |

If `config.yaml` is missing, safe hardcoded defaults are used as a fallback.

In [ ]:
CONFIG = {
    "sample_rate": CFG.get("audio", {}).get("sample_rate", 16000),
    "channels":    CFG.get("audio", {}).get("channels", 1),
    "duration_sec": CFG.get("audio", {}).get("clip_duration_sec", 10.0),
    "machines":    CFG.get("data", {}).get("machine_types", ["fan", "pump", "slider", "valve"]),
    "machine_id":  CFG.get("data", {}).get("machine_ids", ["id_00"])[0],
    "snr":         CFG.get("data", {}).get("snr_levels", ["6_dB"])[0],
}

print("Active Configuration (from config.yaml):")
for k, v in CONFIG.items():
    print(f"  - {k}: {v}")

---
### Step 3: Locate the MIMII Dataset on Disk (Auto-Discovery)

The raw MIMII audio files can live in different locations depending on the runtime:

- **On Kaggle:** The dataset is attached as an input dataset and appears under `/kaggle/input/dc2020task2/`.
- **Locally:** The files are expected inside `data/raw/` relative to the project root.

This cell defines `locate_mimii_data()`, which:
1. Checks the bootstrap-resolved `DATA_ROOT` for `.wav` files using a recursive glob search.
2. Falls back to scanning `/kaggle/input` on Kaggle if the primary path yields no results.
3. Prints the total number of `.wav` files found, or a diagnostic message listing what *is* present if none are found.

The returned `data_root` path is used by every subsequent cell to locate audio files.

In [ ]:
def locate_mimii_data():
    """Locate MIMII audio files using the bootstrap-resolved DATA_ROOT."""
    search_roots = [str(DATA_ROOT)]
    
    # Fallback paths for edge cases
    if ON_KAGGLE:
        search_roots.append("/kaggle/input")
    
    found_root = None
    for root in search_roots:
        if os.path.exists(root):
            wav_files = glob.glob(os.path.join(root, "**/*.wav"), recursive=True)
            if len(wav_files) > 0:
                print(f"✓ Found {len(wav_files)} .wav files in '{root}'")
                found_root = root
                break
    
    if found_root is None:
        print("✗ No audio files detected.")
        print(f"  Searched: {search_roots}")
        if ON_KAGGLE and os.path.exists("/kaggle/input"):
            print("  Kaggle /kaggle/input/ contents:")
            for root, dirs, files in os.walk("/kaggle/input"):
                print(f"    {root} -> {len(files)} files")
                if root.count(os.sep) > 4:  # Limit depth
                    break
    return found_root

data_root = locate_mimii_data()

---
### Step 4: Scan & Index Every Audio File in the Dataset

This is the **core indexing engine** of the notebook. The function `scan_dataset()` walks the entire directory tree under `data_root`, finds every `.wav` file, and classifies it along four dimensions:

| Dimension | How it is detected | Example values |
|---|---|---|
| **Machine Type** | Keywords in the file path (`fan`, `pump`, `slider`, `valve`) | `fan` |
| **Condition** | Presence of `normal`, `abnormal`, or `anomaly` in the path | `normal`, `anomaly` |
| **Machine ID** | Substrings like `id_00`, `id_01`, etc. | `id_00` |
| **SNR Level** | Keywords like `6_dB`, `0_dB`, `-6_dB` in the path | `6_dB` |

**Important parsing detail:** The scanner checks for `abnormal` *before* `normal` to avoid a substring false-match (the word "normal" appears inside "abnormal").

The output is a Pandas DataFrame (`df_all`) with one row per audio file and columns for `file_path`, `filename`, `machine_type`, `machine_id`, `condition`, `snr`, and `file_size_kb`.

In [ ]:
def scan_dataset(root_path):
    """Scan all .wav files under root_path and classify by machine/condition/ID."""
    if root_path is None:
        print("Warning: Root path is None. Please attach the dataset to your notebook.")
        return pd.DataFrame()
        
    records = []
    wav_files = glob.glob(os.path.join(root_path, "**/*.wav"), recursive=True)
    print(f"Scanning {len(wav_files)} audio files...")
    
    for path_str in tqdm(wav_files, desc="Indexing files"):
        p = Path(path_str)
        parts = [part.lower() for part in p.parts]
        filename = p.name.lower()
        
        # Determine machine type
        m_type = "unknown"
        for m in ["fan", "pump", "slider", "valve"]:
            if any(m in part for part in parts) or m in filename:
                m_type = m
                break
        
        # Determine condition (normal vs anomaly/abnormal)
        # IMPORTANT: Check 'abnormal' BEFORE 'normal' to avoid substring false-match
        condition = "unknown"
        if any("abnormal" in part or "anomaly" in part for part in parts) or "abnormal" in filename or "anomaly" in filename:
            condition = "anomaly"
        elif any("normal" in part for part in parts) or "normal" in filename:
            condition = "normal"
            
        # Determine Machine ID
        m_id = "id_00"
        for part in parts:
            if "id_0" in part or "id_1" in part or "id_2" in part:
                for chunk in part.split("_"):
                    if chunk in ["00", "01", "02", "03", "04", "05", "06"]:
                        m_id = f"id_{chunk}"
        for chunk in filename.split("_"):
            if chunk in ["00", "01", "02", "03", "04", "05", "06"]:
                m_id = f"id_{chunk}"

        # Determine SNR if present
        snr = "6_dB"
        if "0_db" in parts or "0_db" in filename or "0db" in filename:
            snr = "0_dB"
        elif "-6_db" in parts or "-6_db" in filename or "min6db" in filename:
            snr = "-6_dB"
        elif "6_db" in parts or "6_db" in filename or "6db" in filename:
            snr = "6_dB"
            
        records.append({
            "file_path": str(p),
            "filename": p.name,
            "machine_type": m_type,
            "machine_id": m_id,
            "condition": condition,
            "snr": snr,
            "file_size_kb": round(p.stat().st_size / 1024, 2)
        })
        
    df = pd.DataFrame(records)
    return df

df_all = scan_dataset(data_root)
if not df_all.empty:
    print(f"Successfully indexed {len(df_all)} audio clips.")
    display(df_all.head())

---
### Step 5: Filter the Dataset to Our Project Scope

The full MIMII dataset contains multiple machine IDs and SNR levels. For this project we narrow the scope to a controlled subset:

- **Machine Types:** `fan`, `pump`, `slider`, `valve` — four fundamentally different industrial machines.
- **Machine ID:** `id_00` — a single physical unit per machine type, ensuring consistency.
- **Conditions:** Both `normal` (healthy operation) and `anomaly` (faulty operation) clips are kept.

#### What the code does
1. Filters `df_all` to keep only the 4 target machine types.
2. For each machine type, selects clips belonging to `id_00` (or the first available ID if `id_00` is absent in the data).
3. Concatenates the filtered subsets into `df_scope`.
4. Builds a **pivot-table summary** showing the number of Normal and Anomalous clips per machine type, plus the percentage of normal clips — this gives an immediate sense of class balance.

In [ ]:
if not df_all.empty:
    target_machines = CONFIG["machines"]
    df_scope = df_all[df_all["machine_type"].isin(target_machines)].copy()
    
    # Filter to id_00 per machine, keep first available ID for machines without id_00
    filtered_parts = []
    for m in target_machines:
        m_df = df_scope[df_scope["machine_type"] == m]
        if m_df.empty:
            continue
        available_ids = sorted(m_df["machine_id"].unique())
        target_id = CONFIG["machine_id"] if CONFIG["machine_id"] in available_ids else available_ids[0]
        filtered_parts.append(m_df[m_df["machine_id"] == target_id])
        if target_id != CONFIG["machine_id"]:
            print(f"  ℹ {m}: id_00 not available, using {target_id}")
    
    df_scope = pd.concat(filtered_parts, ignore_index=True)
        
    print(f"Total clips within scope: {len(df_scope)}")
    
    summary_table = pd.pivot_table(
        df_scope,
        index=["machine_type", "machine_id"],
        columns="condition",
        values="filename",
        aggfunc="count",
        fill_value=0
    ).reset_index()
    
    if "normal" not in summary_table.columns: summary_table["normal"] = 0
    if "anomaly" not in summary_table.columns: summary_table["anomaly"] = 0
    
    summary_table["total_clips"] = summary_table["normal"] + summary_table["anomaly"]
    summary_table["normal_pct"] = (summary_table["normal"] / summary_table["total_clips"] * 100).round(1)
    
    print("\n" + "="*60)
    print("MIMII DATASET INVENTORY SUMMARY")
    print("="*60)
    display(summary_table)

---
### Step 6: Audio File Integrity & Physical Parameter Verification

Even if the files exist and are correctly labelled, they could be **corrupted**, **truncated**, or recorded with **unexpected settings**. This cell performs a rigorous quality check on a stratified sample of audio files.

#### What is verified
| Check | Expected Value | Why it matters |
|---|---|---|
| Sample Rate | 16,000 Hz | All downstream feature extraction assumes this rate; a mismatch would shift frequency content |
| Channels | 1 (mono) | Multi-channel files would need additional handling |
| Duration | ≈ 10 seconds | Clips shorter than ~5 s may indicate truncation or download errors |
| Format | WAV / PCM_16 | Ensures lossless, uncompressed audio |

#### How the sampling works
Rather than checking every file (which can be slow for large datasets), we use **stratified sampling**: 5 random files from each machine type, ensuring we verify at least some clips from every category.

If all sampled files pass, we print a confirmation message. Any failures are flagged with details.

In [ ]:
def verify_audio_integrity(df, samples_per_machine=5):
    """Verify audio integrity with stratified sampling across all machine types."""
    if df.empty:
        print("No files to verify.")
        return
    
    # Stratified sample: pick files from each machine type
    sample_df = df.groupby("machine_type", group_keys=False).apply(
        lambda x: x.sample(min(samples_per_machine, len(x)), random_state=42)
    )
    
    verification_results = []
    print(f"Testing {len(sample_df)} audio clips across {df['machine_type'].nunique()} machine types...")
    
    for idx, row in sample_df.iterrows():
        try:
            info = sf.info(row["file_path"])
            verification_results.append({
                "machine": row["machine_type"],
                "condition": row["condition"],
                "sample_rate": info.samplerate,
                "channels": info.channels,
                "duration_sec": round(info.duration, 2),
                "format": info.format,
                "subtype": info.subtype,
                "is_valid": info.samplerate == 16000 and info.duration > 5.0
            })
        except Exception as e:
            verification_results.append({
                "machine": row["machine_type"],
                "condition": row["condition"],
                "sample_rate": None,
                "channels": None,
                "duration_sec": None,
                "format": "ERROR",
                "subtype": str(e),
                "is_valid": False
            })
            
    res_df = pd.DataFrame(verification_results)
    display(res_df)
    
    all_valid = res_df["is_valid"].all()
    print(f"\nIntegrity Check Passed: {all_valid}")
    if all_valid:
        print("✓ All inspected audio files strictly match 16,000 Hz 16-bit PCM standard!")

if not df_all.empty:
    verify_audio_integrity(df_scope)

---
### Step 7: Export Summary Reports & Visualise Dataset Distribution

This final cell saves two key outputs into the `reports/` directory:

| File | Contents | Used by |
|---|---|---|
| `indexed_dataset.csv` | Full file-level catalog with paths, machine types, conditions, and IDs | Notebook 02 (visualization), Notebook 03 (preprocessing) |
| `data_summary.csv` | Aggregated clip counts per machine type and condition | Quick reference for reporting |

It also generates a **bar chart** (`dataset_distribution.png`) showing the Normal vs. Anomalous clip counts side by side for each machine type. This chart helps you visually confirm:
- Whether the dataset is **class-imbalanced** (in anomaly detection, normal clips typically far outnumber anomalous ones — this is expected and by design).
- Whether any machine type is missing data entirely.

#### What's next?
With verified, cataloged data in hand, proceed to **Notebook 02** for audio waveform and spectrogram exploration, or directly to **Notebook 03** for Mel-spectrogram feature extraction.

In [ ]:
if not df_all.empty:
    # Save full indexed dataset
    indexed_path = os.path.join(OUTPUT_DIR, "indexed_dataset.csv")
    df_scope.to_csv(indexed_path, index=False)
    print(f"Saved: {indexed_path} ({len(df_scope)} records)")
    
    # Save summary table
    summary_path = os.path.join(OUTPUT_DIR, "data_summary.csv")
    summary_table.to_csv(summary_path, index=False)
    print(f"Saved: {summary_path}")
    
    # Plot dataset distribution
    plt.figure(figsize=(9, 4.5))
    machines = summary_table["machine_type"]
    normal_counts = summary_table["normal"]
    anomaly_counts = summary_table["anomaly"]
    
    x = np.arange(len(machines))
    width = 0.35
    
    plt.bar(x - width/2, normal_counts, width, label='Normal (Train/Val/Test)', color='#2b5c8f')
    plt.bar(x + width/2, anomaly_counts, width, label='Anomalous (Test Only)', color='#d9534f')
    
    plt.xlabel('Machine Type', fontsize=11, fontweight='bold')
    plt.ylabel('Clip Count', fontsize=11, fontweight='bold')
    plt.title('MIMII Dataset Distribution per Machine Type (ID 00)', fontsize=13, fontweight='bold')
    plt.xticks(x, [m.upper() for m in machines])
    plt.legend(frameon=True)
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.tight_layout()
    
    chart_path = os.path.join(OUTPUT_DIR, "dataset_distribution.png")
    plt.savefig(chart_path, dpi=300)
    plt.show()
    print(f"Saved distribution chart: {chart_path}")
    
    print("\n" + "="*60)
    print("PHASE 1 - NOTEBOOK 01 COMPLETED SUCCESSFULLY!")
    print("="*60)